**TASK 1**

In [1]:
# Mengimpor Pustaka (Library) yang Dibutuhkan
import geopandas as gpd
import pandas as pd
import numpy as np

In [2]:
# Menghubungkan Google Drive ke Google Colab
from google.colab import drive
drive.mount('/content/drive')

# Mendefinisikan ulang path direktori file
file_path = '/content/drive/MyDrive/Online_Assesment_GISACT/Ekstrak_Final.geojson'

# Memuat data geospasial
import geopandas as gpd
gdf = gpd.read_file(file_path)

print("Data berhasil dimuat. Total observasi:", len(gdf))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data berhasil dimuat. Total observasi: 36640


In [3]:
# PEMBERSIHAN DATA (Handling Missing Values)
# Hapus bangunan yang datanya bolong (null) di LST atau AOD akibat resolusi satelit
gdf = gdf.dropna(subset=['LST', 'AOD', 'GHI'])
print(f"Total bangunan setelah pembersihan: {len(gdf)}")

Total bangunan setelah pembersihan: 36552


In [4]:
# MENGHITUNG LUAS ATAP (METER PERSEGI)
gdf = gdf.to_crs(epsg=32748)
gdf['luas_atap_m2'] = gdf.geometry.area

# Asumsi: Tidak semua atap bisa dipasang panel (ada toren air, genteng miring, dll).
# Menggunakan Useable Area sebesar 60% dari total luas atap.
gdf['luas_efektif_m2'] = gdf['luas_atap_m2'] * 0.60

# MENGHITUNG POTENSI ENERGI (TASK 1)
# Mengonversi GHI Harian menjadi tahunan
gdf['GHI_tahunan'] = gdf['GHI'] * 365

# Parameter Panel Surya
efisiensi = 0.15      # 15%
perf_ratio = 0.75     # 75%

# Rumus Energi: E = A * r * H * PR
gdf['potensi_energi_kwh_thn'] = (
    gdf['luas_efektif_m2'] * gdf['GHI_tahunan'] * efisiensi * perf_ratio
)

# Test Hasil Task 1
print("\n--- HASIL TASK 1: 5 BANGUNAN DENGAN POTENSI ENERGI TERTINGGI ---")
task1_result = gdf[['id', 'luas_atap_m2', 'GHI_tahunan', 'potensi_energi_kwh_thn']].sort_values(by='potensi_energi_kwh_thn', ascending=False)
print(task1_result.head())


--- HASIL TASK 1: 5 BANGUNAN DENGAN POTENSI ENERGI TERTINGGI ---
                     id  luas_atap_m2  GHI_tahunan  potensi_energi_kwh_thn
36509     way/121706055  38304.682940  1657.720451            4.286146e+06
36395     way/538319726  31436.411320  1660.134428            3.522735e+06
8678      way/974882100  30214.719662  1657.699977            3.380868e+06
35610  relation/9627236  20836.518532  1696.036926            2.385417e+06
36411     way/498751669  14622.431337  1655.072355            1.633580e+06


**TASK 2**

In [5]:
# =========================================================================
# TASK 2: KELAYAKAN EKONOMI & RETURN ON INVESTMENT
# =========================================================================

# Estimasi Kapasitas Sistem (kWp)
# 1 m2 panel surya efisiensi 15% ekuivalen dengan 0.15 kWp
gdf['kapasitas_sistem_kwp'] = gdf['luas_efektif_m2'] * 0.15

# Parameter Finansial
biaya_instalasi_per_kwp = 15000000  # CAPEX: Rp 15 Juta / kWp
tarif_listrik_pln = 1444.70         # Penghematan: Tarif R-1 (Rp/kWh)

# Kalkulasi CAPEX, Savings, dan Payback Period
gdf['biaya_investasi_rp'] = gdf['kapasitas_sistem_kwp'] * biaya_instalasi_per_kwp
gdf['penghematan_tahunan_rp'] = gdf['potensi_energi_kwh_thn'] * tarif_listrik_pln
gdf['waktu_balik_modal_tahun'] = gdf['biaya_investasi_rp'] / gdf['penghematan_tahunan_rp']

print("[SUCCESS] Task 2 Selesai. Payback Period terhitung.")

[SUCCESS] Task 2 Selesai. Payback Period terhitung.


**TASK 3**

In [6]:
# =========================================================================
# TASK 3: ANALISIS DEGRADASI LINGKUNGAN (SUHU & POLUSI)
# =========================================================================
from sklearn.preprocessing import MinMaxScaler

# A. Standardisasi Skala (0 - 1) untuk Suhu (LST) dan Polusi (AOD)
scaler_lingkungan = MinMaxScaler()
gdf[['LST_norm', 'AOD_norm']] = scaler_lingkungan.fit_transform(gdf[['LST', 'AOD']])

# B. Formulasi Environmental Degradation Index (EDI)
# Bobot seimbang 50:50 antara paparan panas dan kualitas udara
gdf['indeks_degradasi'] = (gdf['LST_norm'] + gdf['AOD_norm']) / 2

print("[SUCCESS] Task 3 Selesai. Indeks Degradasi terhitung.")

[SUCCESS] Task 3 Selesai. Indeks Degradasi terhitung.


In [7]:
# =========================================================================
# TASK 4: MACHINE LEARNING, SMCE, & EKSPOR FINAL
# =========================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# A. K-Means Clustering (Machine Learning)
fitur_ml = ['potensi_energi_kwh_thn', 'waktu_balik_modal_tahun', 'indeks_degradasi', 'jarak_jala']
scaler_ml = StandardScaler()
X_scaled = scaler_ml.fit_transform(gdf[fitur_ml])

# Clustering menjadi 3 klasifikasi (Prioritas Tinggi, Sedang, Rendah)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
gdf['cluster'] = kmeans.fit_predict(X_scaled)

# B. Spatial Multi-Criteria Evaluation (SMCE) - Skoring Akhir
scaler_skor = MinMaxScaler()
gdf[['n_energi', 'n_roi', 'n_jarak']] = scaler_skor.fit_transform(gdf[['potensi_energi_kwh_thn', 'waktu_balik_modal_tahun', 'jarak_jala']])

# Inversi nilai: ROI & Jarak yang kecil = Lebih Baik (1 - x)
gdf['n_roi'] = 1 - gdf['n_roi']
gdf['n_jarak'] = 1 - gdf['n_jarak']

# Aplikasi Bobot Prioritas
bobot = {'energi': 0.30, 'roi': 0.30, 'lingkungan': 0.25, 'jarak': 0.15}
gdf['skor_prioritas_akhir'] = (
    (gdf['n_energi'] * bobot['energi']) +
    (gdf['n_roi'] * bobot['roi']) +
    (gdf['indeks_degradasi'] * bobot['lingkungan']) +
    (gdf['n_jarak'] * bobot['jarak'])
)

# Sorting dari skor prioritas tertinggi
gdf = gdf.sort_values(by='skor_prioritas_akhir', ascending=False)

# C. EKSPOR FILE FINAL
gdf.drop(columns='geometry').to_csv('LAPORAN_FINAL_PRIORITAS.csv', index=False)
gdf.to_file('PETA_FINAL_PRIORITAS.geojson', driver='GeoJSON')

print("[SUCCESS] Task 4 Selesai. Data Master berhasil diekspor!")

[SUCCESS] Task 4 Selesai. Data Master berhasil diekspor!
